In [0]:
   %pip install kaggle py7zr

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 87.0 MB/s  0:00:00
  Attempting uninstall: certifi
    Found existing installation: certifi 2025.11.12
    Not uninstalling certifi at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7b05f9b7-b581-47a0-bfd8-7916d6065011
    Can't uninstall 'certifi'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [kaggle]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
os.environ["KAGGLE_USERNAME"] = "fatima2709"
os.environ["KAGGLE_KEY"] = "removed"

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

raw = "/Volumes/workspace/default/kkbox_raw"
api.competition_download_file("kkbox-churn-prediction-challenge", "transactions.csv.7z", path=raw)

transactions.csv.7z: Skipping, found more recently modified local copy (use --force to force download)


In [0]:
import py7zr
py7zr.SevenZipFile(raw + "/transactions.csv.7z").extractall(raw)

for root, dirs, files in os.walk(raw):
    if "transactions.csv" in files:
        print(root)

/Volumes/workspace/default/kkbox_raw


In [0]:
import pyspark.sql.functions as F

history = spark.read.csv(raw + "/transactions.csv", header=True, inferSchema=True)
history = history.filter(F.col("transaction_date") < 20170301)

pay_before = history.groupBy("msno").agg(
    F.avg("payment_plan_days").alias("plan_days"),
    F.avg("is_auto_renew").alias("auto_renew"),
    F.avg(F.col("plan_list_price") - F.col("actual_amount_paid")).alias("avg_discount"),
    F.max("is_cancel").alias("ever_cancelled"),
    F.count("*").alias("num_payments"),
    F.sum("actual_amount_paid").alias("total_paid")
)

In [0]:
pay_before.count()

2363626

In [0]:
customers = spark.table("workspace.default.customers_clean")
customers = customers.drop("plan_days", "auto_renew", "avg_discount", "ever_cancelled", "num_payments", "total_paid")
customers = customers.join(pay_before, on="msno", how="left")

In [0]:
covered = customers.filter(F.col("num_payments").isNotNull()).count()
print("customers with pre-March payments:", covered)
print("%:", round(covered / customers.count() * 100, 1))

customers with pre-March payments: 968433
%: 99.7


In [0]:
customers.coalesce(1).write.mode("overwrite").option("header", True).csv(raw + "/customers_clean_v2")